In [1]:
# Test 1: Check Python environment
import sys
print(f"Python version: {sys.version}")
print(f"Python path: {sys.executable}")

Python version: 3.11.11 | packaged by conda-forge | (main, Dec  5 2024, 14:17:24) [GCC 13.3.0]
Python path: /opt/conda/bin/python3.11


In [ ]:
# Test 2: Check CUDA
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")



PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4080 Laptop GPU
CUDA memory allocated: 0.00 GB


In [ ]:

# Test 3: Check key imports
import numpy as np
import pandas as pd
import transformers
import sentence_transformers
import llama_index
import langchain

print(f"\nNumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Sentence Transformers: {sentence_transformers.__version__}")
print(f"LlamaIndex: {llama_index.__version__}")
print(f"LangChain: {langchain.__version__}")

In [ ]:
import httpx
import json
import numpy as np
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

# Test all services
print("Testing all services...")
services = {
    "LLM Server": "http://localhost:8001/health",
    "Embedding Server": "http://localhost:8002/health",
    "Qdrant": "http://localhost:6333"
}

for name, url in services.items():
    try:
        if name == "Qdrant":
            resp = httpx.get(url, timeout=5)
            status = "✓" if resp.status_code == 200 else "✗"
        else:
            resp = httpx.get(url, timeout=5)
            data = resp.json()
            status = "✓" if data.get("status") == "healthy" else "✗"
        print(f"{name}: {status}")
    except Exception as e:
        print(f"{name}: ✗ ({str(e)[:50]})")

# Load and prepare sample data
print("\nLoading sample data...")
with open('/workspace/data/fiqa/sample.jsonl', 'r') as f:
    data = [json.loads(line) for line in f]

df = pd.DataFrame(data)
print(f"Loaded {len(df)} samples")

# Setup Qdrant
print("\nSetting up Qdrant...")
client = QdrantClient(host="localhost", port=6333, timeout=10)

collection_name = "fiqa_documents"
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )

# Generate embeddings and upload
print("Generating embeddings...")
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

points = []
for idx, row in df.iterrows():
    embedding = embedding_model.encode(row['context'])
    points.append(PointStruct(
        id=idx,
        vector=embedding.tolist(),
        payload={
            "text": row['context'],
            "question": row['question'],
            "answer": row['answer'],
            "source": "fiqa_sample"
        }
    ))

# Upload to Qdrant
client.upsert(collection_name=collection_name, points=points)
print(f"Uploaded {len(points)} documents")

# Test retrieval
print("\nTesting retrieval...")
test_query = "What is inflation?"
query_embedding = embedding_model.encode(test_query).tolist()

results = client.search(
    collection_name=collection_name,
    query_vector=query_embedding,
    limit=3
)

print(f"Query: {test_query}")
for i, result in enumerate(results):
    print(f"\n{i+1}. Score: {result.score:.3f}")
    print(f"   Context: {result.payload['text'][:100]}...")

# Test LLM generation
print("\nTesting LLM with RAG...")
context = "\n".join([f"- {r.payload['text']}" for r in results])
prompt = f"""Based on the following context, answer the question.

Context:
{context}

Question: {test_query}

Answer:"""

resp = httpx.post(
    "http://localhost:8001/generate",
    json={"prompt": prompt, "max_tokens": 150, "temperature": 0.7}
)

if resp.status_code == 200:
    answer = resp.json()['text']
    print(f"\nGenerated answer: {answer}")
else:
    print(f"LLM error: {resp.status_code}")

print("\n✅ Setup complete! Ready for RAGAS evaluation.")